In [20]:
# call series of multiple LLM's
# you decompose your task
# In this workflow, we will generate a LLM outline from topic, and then we'll send topic + LLM outline to generate blog

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import  TypedDict
import os
from dotenv import load_dotenv
load_dotenv()


True

In [22]:
model =  ChatGroq(
    model="llama-3.1-8b-instant",   # or "mixtral-8x7b-32768", "gemma2-9b-it"
    temperature=0.7,
    api_key=os.environ["GROQ_API_KEY"]
)


In [23]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation: str



In [24]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f'Generate a detailed outline for a blog on the topic'
    outline = model.invoke(prompt).content

    state['outline'] = outline
    return  state

In [25]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the title - ${title} using the following outline \n ${outline}'
    content = model.invoke(prompt).content
    state['content'] = content

    return state

In [26]:
def evaluate(state: BlogState) -> BlogState:
    title = state['title']
    blog_content = state['content']
    prompt = f'Evaluate my blog on the topic - title: ${title} \n Blog: {blog_content}'
    evaluation = model.invoke(prompt).content
    state['evaluation'] = evaluation
    return state



In [27]:
graph = StateGraph(BlogState)
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

workflow = graph.compile()

In [28]:
initial_state = {'title': 'Rise of AI in india'}
final_state = workflow.invoke(initial_state)
# print(final_state)
# print(final_state['outline'])
print(final_state['content'])
print("Here is the evaluation of your blog:\n")
print(final_state['evaluation'])

**Rise of AI in India: Unlocking Opportunities and Challenges**

**I. Introduction**

Are you tired of feeling overwhelmed and struggling to keep up with the rapid pace of technological advancements in India? The rise of Artificial Intelligence (AI) in India has brought about numerous opportunities for growth and innovation, but it also poses significant challenges for businesses, governments, and individuals alike. In this blog post, we'll explore the rise of AI in India, its applications, benefits, and challenges, and provide insights on how to navigate this exciting and complex landscape.

**II. Understanding the AI Landscape in India**

India has made significant strides in AI research and development, with the government announcing plans to invest $1 billion in AI research and development over the next five years. The country is also home to a growing number of AI startups, with many of them focusing on applications such as healthcare, finance, and education.

**III. Applications 